# Загрузка библиотек


In [1]:
import pandas as pd
import numpy as np

In [2]:
# Библиотеки для работы Colab с Google Sheets
from google.colab import auth
import gspread
from google.auth import default

In [3]:
# Авторизация пользвателя
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

print("Авторизация пройдена! Библиотеки загружены.")

Авторизация пройдена! Библиотеки загружены.


# Загрузка и отчистка данных

In [4]:
# Загружаю данные через открытый доступ Google Драйва.
# https://drive.google.com/file/d/145IohW5nU0j2cnwqknew1zr0T64JVFGy/view?usp=drive_link - fin
# https://drive.google.com/file/d/1SqN5ybR0BjprpEnf1s10pQXNXE6RAS7_/view?usp=drive_link - prol

!gdown --id 145IohW5nU0j2cnwqknew1zr0T64JVFGy
!gdown --id 1SqN5ybR0BjprpEnf1s10pQXNXE6RAS7_

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=145IohW5nU0j2cnwqknew1zr0T64JVFGy
To: /content/financial_data.csv
100% 74.3k/74.3k [00:00<00:00, 111MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1SqN5ybR0BjprpEnf1s10pQXNXE6RAS7_
To: /content/prolongations.csv
100% 35.7k/35.7k [00:00<00:00, 102MB/s]


In [5]:
df_fin_raw = pd.read_csv('/content/financial_data.csv')
df_prol_raw = pd.read_csv('/content/prolongations.csv')

In [6]:
# 2. Переводим финансы из широкого формата в длинный (Unpivot)
id_vars = ['id', 'Account']
month_cols = [col for col in df_fin_raw.columns if col not in id_vars and col != 'Причина дубля']
df_fin_long = pd.melt(df_fin_raw, id_vars=id_vars, value_vars=month_cols, var_name='month_str', value_name='amount_raw')
df_fin_long = df_fin_long.dropna(subset=['amount_raw']) # Удаляем пустые ячейки

In [7]:
display(df_fin_long.info(), df_fin_long.head(3))

<class 'pandas.core.frame.DataFrame'>
Index: 2596 entries, 0 to 7215
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          2596 non-null   int64 
 1   Account     2596 non-null   object
 2   month_str   2596 non-null   object
 3   amount_raw  2596 non-null   object
dtypes: int64(1), object(3)
memory usage: 101.4+ KB


None

,id,Account,month_str,amount_raw
0,42,Васильев Артем Александрович,Ноябрь 2022,"36 220,00"
1,657,Васильев Артем Александрович,Ноябрь 2022,стоп
2,657,Васильев Артем Александрович,Ноябрь 2022,стоп


In [8]:
# 3. Функция очистки денег (убираем неразрывные пробелы и статусы)
def clean_money(val):
    if pd.isna(val) or str(val).lower() in ['стоп', 'в ноль', 'end']:
        return 0.0
    s = str(val).replace('\xa0', '').replace(' ', '').replace(',', '.')
    try:
        return float(s)
    except:
        return 0.0

# Создаем еще одну колонку с корректными значениями
df_fin_long['amount'] = df_fin_long['amount_raw'].apply(clean_money)

In [9]:
display(df_fin_long.info(), df_fin_long.head(3))

<class 'pandas.core.frame.DataFrame'>
Index: 2596 entries, 0 to 7215
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          2596 non-null   int64  
 1   Account     2596 non-null   object 
 2   month_str   2596 non-null   object 
 3   amount_raw  2596 non-null   object 
 4   amount      2596 non-null   float64
dtypes: float64(1), int64(1), object(3)
memory usage: 121.7+ KB


None

,id,Account,month_str,amount_raw,amount
0,42,Васильев Артем Александрович,Ноябрь 2022,"36 220,00",36220.0
1,657,Васильев Артем Александрович,Ноябрь 2022,стоп,0.0
2,657,Васильев Артем Александрович,Ноябрь 2022,стоп,0.0


In [10]:
# 4. Функция перевода русских месяцев в даты
ru_months = {'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4, 'май': 5, 'июнь': 6,
             'июль': 7, 'август': 8, 'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12}

def parse_date(s):
    s = str(s).lower().strip()
    try:
        parts = s.split(' ')
        return pd.Timestamp(year=int(parts[1]), month=ru_months[parts[0]], day=1)
    except:
        return pd.NaT

df_fin_long['date'] = df_fin_long['month_str'].apply(parse_date)
df_prol_raw['end_date'] = df_prol_raw['month'].apply(parse_date)

In [11]:
display(df_fin_long.info(), df_fin_long.head(3))
display(df_prol_raw.info(), df_prol_raw.head(3))


<class 'pandas.core.frame.DataFrame'>
Index: 2596 entries, 0 to 7215
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          2596 non-null   int64         
 1   Account     2596 non-null   object        
 2   month_str   2596 non-null   object        
 3   amount_raw  2596 non-null   object        
 4   amount      2596 non-null   float64       
 5   date        2596 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 142.0+ KB


None

,id,Account,month_str,amount_raw,amount,date
0,42,Васильев Артем Александрович,Ноябрь 2022,"36 220,00",36220.0,2022-11-01
1,657,Васильев Артем Александрович,Ноябрь 2022,стоп,0.0,2022-11-01
2,657,Васильев Артем Александрович,Ноябрь 2022,стоп,0.0,2022-11-01


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   id        477 non-null    int64         
 1   month     477 non-null    object        
 2   AM        477 non-null    object        
 3   end_date  477 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 15.0+ KB


None

,id,month,AM,end_date
0,42,ноябрь 2022,Васильев Артем Александрович,2022-11-01
1,453,ноябрь 2022,Васильев Артем Александрович,2022-11-01
2,548,ноябрь 2022,Михайлов Андрей Сергеевич,2022-11-01


In [12]:
# 5. Группируем дубликаты (схлопываем "первые/вторые части" платежей)
df_fin = df_fin_long.groupby(['id', 'date'])['amount'].sum().reset_index()
df_prol = df_prol_raw.groupby(['id', 'end_date'])['AM'].first().reset_index()

print(f"Данные очищены! Уникальных проектов: {df_prol['id'].nunique()}")

Данные очищены! Уникальных проектов: 313


# Расчет бизнес-мекрик по ТЗ.

In [13]:
# Формируем витрину данных, вычисляя M+1 (первый месяц) и M+2 (второй месяц) для каждого проекта.

# Создаем "сдвинутые" даты для проверки оплат в будущем
df_prol['date_m1'] = df_prol['end_date'] + pd.DateOffset(months=1)
df_prol['date_m2'] = df_prol['end_date'] + pd.DateOffset(months=2)

In [14]:
# Подтягиваем базовую выручку (за месяц завершения M)
master = pd.merge(df_prol, df_fin, left_on=['id', 'end_date'], right_on=['id', 'date'], how='left')
master = master.rename(columns={'amount': 'rev_base'}).drop(columns=['date'])

In [15]:
# Подтягиваем выручку первого месяца (M+1)
master = pd.merge(master, df_fin, left_on=['id', 'date_m1'], right_on=['id', 'date'], how='left')
master = master.rename(columns={'amount': 'rev_m1'}).drop(columns=['date'])

In [16]:
# Подтягиваем выручку второго месяца (M+2)
master = pd.merge(master, df_fin, left_on=['id', 'date_m2'], right_on=['id', 'date'], how='left')
master = master.rename(columns={'amount': 'rev_m2'}).drop(columns=['date'])

In [17]:
# Заполняем пропуски нулями
master[['rev_base', 'rev_m1', 'rev_m2']] = master[['rev_base', 'rev_m1', 'rev_m2']].fillna(0)

In [18]:
master

,id,end_date,AM,date_m1,date_m2,rev_base,rev_m1,rev_m2
0,15,2022-12-01,Иванова Мария Сергеевна,2023-01-01,2023-02-01,439280.00,102433.75,102433.75
1,15,2023-02-01,Иванова Мария Сергеевна,2023-03-01,2023-04-01,102433.75,102433.75,138158.00
2,15,2023-03-01,Иванова Мария Сергеевна,2023-04-01,2023-05-01,102433.75,138158.00,138158.00
3,15,2023-04-01,Иванова Мария Сергеевна,2023-05-01,2023-06-01,138158.00,138158.00,102433.75
4,15,2023-06-01,Иванова Мария Сергеевна,2023-07-01,2023-08-01,102433.75,0.00,0.00
...,...,...,...,...,...,...,...,...
468,1001,2023-11-01,Кузнецов Михаил Иванович,2023-12-01,2024-01-01,290000.00,0.00,0.00
469,1004,2023-12-01,без А/М,2024-01-01,2024-02-01,0.00,0.00,0.00
470,1006,2023-12-01,Смирнова Ольга Владимировна,2024-01-01,2024-02-01,140495.00,0.00,0.00
471,1012,2023-11-01,Петрова Анна Дмитриевна,2023-12-01,2024-01-01,98492.00,109442.52,0.00


In [19]:
# Применяем условия из ТЗ
# База для 2-го месяца: берем в расчет только те проекты, где в 1-й месяц не было оплат
master['rev_base_m2'] = np.where(master['rev_m1'] == 0, master['rev_base'], 0)

In [20]:
# Фильтруем данные: оставляем только проекты, где M+1 или M+2 попадают на 2023 год
master_2023 = master[
    (master['date_m1'].dt.year == 2023) | (master['date_m2'].dt.year == 2023)
].copy()

print("Бизнес-логика применена. Подготовлена мастер-таблица.")

Бизнес-логика применена. Подготовлена мастер-таблица.


# Сборка витрины (CSV) для DataLens

In [21]:
# Группируем по Менеджеру и Месяцу события (M+1)
dl_m1 = master_2023[master_2023['date_m1'].dt.year == 2023].groupby(['AM', 'date_m1']).agg(
    base_sum=('rev_base', 'sum'),
    fact_sum=('rev_m1', 'sum')
).reset_index()
dl_m1['type'] = 'Пролонгация 1 месяц'
dl_m1 = dl_m1.rename(columns={'date_m1': 'date'})

# Группируем по Менеджеру и Месяцу события (M+2)
dl_m2 = master_2023[master_2023['date_m2'].dt.year == 2023].groupby(['AM', 'date_m2']).agg(
    base_sum=('rev_base_m2', 'sum'),
    fact_sum=('rev_m2', 'sum')
).reset_index()
dl_m2['type'] = 'Пролонгация 2 месяц'
dl_m2 = dl_m2.rename(columns={'date_m2': 'date'})

# Объединяем в одну таблицу
df_datalens = pd.concat([dl_m1, dl_m2], ignore_index=True)
df_datalens['date'] = df_datalens['date'].dt.strftime('%Y-%m-%d') # Формат для БД

# Сохраняем в CSV (этот файл можно будет подгрузить в Yandex DataLens)
df_datalens.to_csv('datalens_export_2023.csv', index=False)
print("Витрина для DataLens готова и сохранена как datalens_export_2023.csv!")

Витрина для DataLens готова и сохранена как datalens_export_2023.csv!


# Sheets


In [22]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

# ==========================================
# 1. ПОДГОТОВКА ДАННЫХ
# ==========================================
# Берем нашу готовую таблицу master_2023 из предыдущих шагов
df_prep = master_2023.copy()

# Создаем единую колонку с датой (месяцем), когда произошла пролонгация
df_prep['month_dt'] = df_prep['date_m1'].combine_first(df_prep['date_m2'])
df_prep = df_prep[df_prep['month_dt'].dt.year == 2023]

# ==========================================
# 2. ИНИЦИАЛИЗАЦИЯ И СТИЛИ GOOGLE SHEETS / EXCEL
# ==========================================
wb = Workbook()

# Цвета в точности как на скриншотах (светло-желтый и желто-оранжевый)
fill_header_main = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
fill_header_sub = PatternFill(start_color="FFE699", end_color="FFE699", fill_type="solid")
fill_coeff_title = PatternFill(start_color="FFD966", end_color="FFD966", fill_type="solid")

font_bold = Font(bold=True)
align_center = Alignment(horizontal='center', vertical='center', wrap_text=True)

# Правильная настройка границ (Border)
thin_border = Border(left=Side(style='thin'),
                     right=Side(style='thin'),
                     top=Side(style='thin'),
                     bottom=Side(style='thin'))

def apply_header_style(ws, row, col, fill):
    cell = ws.cell(row=row, column=col)
    cell.fill = fill
    cell.font = font_bold
    cell.alignment = align_center
    cell.border = thin_border

# ==========================================
# ЛИСТ 1: "Весь отдел"
# ==========================================
ws1 = wb.active
ws1.title = "Весь отдел"

# Шапка
ws1.merge_cells('A1:A2')
ws1.cell(1, 1, "Месяц")
ws1.merge_cells('B1:D1')
ws1.cell(1, 2, "Пролонгации в первый месяц")
ws1.merge_cells('E1:G1')
ws1.cell(1, 5, "Пролонгации через месяц")

sub_headers = ["к пролонгации", "пролонгировано", "Коэффициент", "к пролонгации", "пролонгировано", "Коэффициент"]
for i, h in enumerate(sub_headers, start=2):
    ws1.cell(2, i, h)

# Стилизуем шапку
for c in range(1, 8):
    apply_header_style(ws1, 1, c, fill_header_main)
    if c > 1: apply_header_style(ws1, 2, c, fill_header_main)

# Данные по отделу (группировка по месяцам)
dept_data = df_prep.groupby(df_prep['month_dt'].dt.strftime('%m.%Y')).agg(
    b1=('rev_base', 'sum'), f1=('rev_m1', 'sum'),
    b2=('rev_base_m2', 'sum'), f2=('rev_m2', 'sum')
).sort_index()

row_idx = 3
for month, row in dept_data.iterrows():
    ws1.cell(row_idx, 1, month).border = thin_border
    ws1.cell(row_idx, 2, row['b1']).border = thin_border
    ws1.cell(row_idx, 3, row['f1']).border = thin_border
    ws1.cell(row_idx, 4, f"=IF(B{row_idx}=0,0,C{row_idx}/B{row_idx})").number_format = '0.00%'
    ws1.cell(row_idx, 5, row['b2']).border = thin_border
    ws1.cell(row_idx, 6, row['f2']).border = thin_border
    ws1.cell(row_idx, 7, f"=IF(E{row_idx}=0,0,F{row_idx}/E{row_idx})").number_format = '0.00%'

    for c in [4, 7]:
        ws1.cell(row_idx, c).border = thin_border
    row_idx += 1

# ==========================================
# ЛИСТ 2: "Менеджеры за год"
# ==========================================
ws2 = wb.create_sheet("Менеджеры за год")

# Шапка (такая же, но первый столбец "Менеджер")
ws2.merge_cells('A1:A2')
ws2.cell(1, 1, "Менеджер")
ws2.merge_cells('B1:D1')
ws2.cell(1, 2, "Пролонгации в первый месяц")
ws2.merge_cells('E1:G1')
ws2.cell(1, 5, "Пролонгации через месяц")

for i, h in enumerate(sub_headers, start=2):
    ws2.cell(2, i, h)

for c in range(1, 8):
    apply_header_style(ws2, 1, c, fill_header_main)
    if c > 1: apply_header_style(ws2, 2, c, fill_header_main)

# Данные за год (группировка по ФИО)
mgr_year_data = df_prep.groupby('AM').agg(
    b1=('rev_base', 'sum'), f1=('rev_m1', 'sum'),
    b2=('rev_base_m2', 'sum'), f2=('rev_m2', 'sum')
).sort_index()

row_idx = 3
for mgr, row in mgr_year_data.iterrows():
    ws2.cell(row_idx, 1, mgr).border = thin_border
    ws2.cell(row_idx, 2, row['b1']).border = thin_border
    ws2.cell(row_idx, 3, row['f1']).border = thin_border
    ws2.cell(row_idx, 4, f"=IF(B{row_idx}=0,0,C{row_idx}/B{row_idx})").number_format = '0.00%'
    ws2.cell(row_idx, 5, row['b2']).border = thin_border

    # ИСПРАВЛЕННАЯ СТРОКА:
    ws2.cell(row_idx, 6, row['f2']).border = thin_border

    ws2.cell(row_idx, 7, f"=IF(E{row_idx}=0,0,F{row_idx}/E{row_idx})").number_format = '0.00%'

    for c in range(1, 8):
        ws2.cell(row_idx, c).border = thin_border
    row_idx += 1

# ==========================================
# ЛИСТ 3: "Менеджеры по месяцам" (Матрица)
# ==========================================
ws3 = wb.create_sheet("Менеджеры по месяцам")
months_list = sorted(df_prep['month_dt'].dt.strftime('%m.%Y').unique())
managers = sorted(df_prep['AM'].unique())

# Секция 1: Коэффициент 1
ws3.merge_cells('A1:B1')
ws3.cell(1, 1, "Коэффициент 1").fill = fill_coeff_title
ws3.cell(1, 1).font = font_bold

ws3.merge_cells('A2:B2')
ws3.cell(2, 1, "Менеджер")
apply_header_style(ws3, 2, 1, fill_header_main)

# Шапка месяцев
for i, m in enumerate(months_list, start=3):
    ws3.cell(2, i, m)
    apply_header_style(ws3, 2, i, fill_header_main)

row_idx = 3
# Заполнение процентов для Коэфф 1
for mgr in managers:
    ws3.merge_cells(start_row=row_idx, start_column=1, end_row=row_idx, end_column=2)
    ws3.cell(row_idx, 1, mgr).border = thin_border

    for col_idx, m in enumerate(months_list, start=3):
        cell_data = df_prep[(df_prep['AM'] == mgr) & (df_prep['month_dt'].dt.strftime('%m.%Y') == m)]
        b1 = cell_data['rev_base'].sum()
        f1 = cell_data['rev_m1'].sum()

        val = f1 / b1 if b1 > 0 else 0
        cell = ws3.cell(row_idx, col_idx, val)
        cell.number_format = '0.00%'
        cell.border = thin_border
    row_idx += 1

# Отступ
row_idx += 3

# Секция 2: Коэффициент 2
ws3.merge_cells(start_row=row_idx, start_column=1, end_row=row_idx, end_column=2)
ws3.cell(row_idx, 1, "Коэффициент 2").fill = fill_coeff_title
ws3.cell(row_idx, 1).font = font_bold

ws3.merge_cells(start_row=row_idx+1, start_column=1, end_row=row_idx+1, end_column=2)
ws3.cell(row_idx+1, 1, "Менеджер")
apply_header_style(ws3, row_idx+1, 1, fill_header_main)

for i, m in enumerate(months_list, start=3):
    ws3.cell(row_idx+1, i, m)
    apply_header_style(ws3, row_idx+1, i, fill_header_main)

row_idx += 2
for mgr in managers:
    ws3.merge_cells(start_row=row_idx, start_column=1, end_row=row_idx, end_column=2)
    ws3.cell(row_idx, 1, mgr).border = thin_border

    for col_idx, m in enumerate(months_list, start=3):
        cell_data = df_prep[(df_prep['AM'] == mgr) & (df_prep['month_dt'].dt.strftime('%m.%Y') == m)]
        b2 = cell_data['rev_base_m2'].sum()
        f2 = cell_data['rev_m2'].sum()

        val = f2 / b2 if b2 > 0 else 0
        cell = ws3.cell(row_idx, col_idx, val)
        cell.number_format = '0.00%'
        cell.border = thin_border
    row_idx += 1

# Финальное выравнивание ширины колонок
for ws in [ws1, ws2]:
    ws.column_dimensions['A'].width = 30
    for col in ['B', 'C', 'D', 'E', 'F', 'G']:
        ws.column_dimensions[col].width = 18

ws3.column_dimensions['A'].width = 15
ws3.column_dimensions['B'].width = 15
for c in range(3, 15):
    ws3.column_dimensions[chr(64+c)].width = 12

wb.save('Отчет_Пролонгации_ФИНАЛ.xlsx')
print("Готово! Файл 'Отчет_Пролонгации_ФИНАЛ.xlsx' сгенерирован без ошибок.")

Готово! Файл 'Отчет_Пролонгации_ФИНАЛ.xlsx' сгенерирован без ошибок.
